# Dataset zawodnik-mecz dla wszystkich mecz?w

Ten notebook buduje g??wn? tabel? analityczn? projektu. Jeden wiersz oznacza wyst?p jednego zawodnika w jednym meczu.

Tabela ??czy:
- cechy zawodnika per 90 minut,
- kontekst meczu,
- wynik,
- rywala,
- momentum dru?yny,
- xG dru?yny.


In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print("Project root:", PROJECT_ROOT)


## Importy


In [ ]:
import pandas as pd

from mgr_mk.data_loader import load_world_cup_2018_matches
from mgr_mk.datasets import build_player_match_dataset


## Konfiguracja

Domy?lnie notebook liczy wszystkie mecze Mundialu 2018. Je?li chcesz szybki test, ustaw `RUN_ALL_MATCHES = False` i wybierz liczb? mecz?w w `NUMBER_OF_MATCHES_FOR_TEST`.


In [ ]:
RUN_ALL_MATCHES = True
NUMBER_OF_MATCHES_FOR_TEST = 10

OUTPUT_PATH = PROJECT_ROOT / "outputs" / "tables" / "player_match_dataset.csv"


## Lista mecz?w


In [ ]:
df_matches = load_world_cup_2018_matches()

if RUN_ALL_MATCHES:
    match_ids = df_matches["match_id"].tolist()
else:
    match_ids = df_matches.head(NUMBER_OF_MATCHES_FOR_TEST)["match_id"].tolist()

print("Liczba mecz?w do przeliczenia:", len(match_ids))
df_matches[["match_id", "match_date", "home_team", "away_team", "home_score", "away_score", "competition_stage"]].head()


## Budowa datasetu

Ta kom?rka przechodzi po meczach, ?aduje eventy i buduje tabel? zawodnik-mecz.


In [ ]:
player_match_dataset = build_player_match_dataset(
    df_matches,
    match_ids=match_ids,
    window=5,
)

print("Rozmiar datasetu:", player_match_dataset.shape)
print("Liczba unikalnych zawodnik?w:", player_match_dataset["player.id"].nunique())
print("Liczba dru?yn:", player_match_dataset["team.name"].nunique())

player_match_dataset.head(10)


## Kolumny datasetu


In [ ]:
pd.DataFrame({
    "column": player_match_dataset.columns,
    "non_null": [player_match_dataset[col].notna().sum() for col in player_match_dataset.columns],
    "dtype": [player_match_dataset[col].dtype for col in player_match_dataset.columns],
})


## Kontrola jako?ci

Szybkie sprawdzenie, czy ka?dy mecz ma sensown? liczb? zawodnik?w oraz czy wynik i rywal s? przypisane do ka?dego wiersza.


In [ ]:
quality_by_match = (
    player_match_dataset.groupby("match_id")
    .agg(
        players=("player.id", "nunique"),
        teams=("team.name", "nunique"),
        missing_opponent=("opponent", lambda value: value.isna().sum()),
        missing_result=("result", lambda value: value.isna().sum()),
    )
    .reset_index()
)

quality_by_match.head(15)


## Przyk?adowe podsumowania


In [ ]:
player_match_dataset.groupby("team.name").agg(
    matches=("match_id", "nunique"),
    player_match_rows=("player.id", "count"),
    avg_team_xg=("team_xg", "mean"),
    avg_team_momentum=("team_momentum", "mean"),
).sort_values("avg_team_momentum", ascending=False).head(10)


In [ ]:
player_match_dataset[
    [
        "match_id",
        "player.name",
        "team.name",
        "opponent",
        "result",
        "minutes_played",
        "passes_per90",
        "shots_per90",
        "total_xg_per90",
        "team_momentum",
        "team_xg",
    ]
].sort_values("total_xg_per90", ascending=False).head(15)


## Zapis do CSV

Plik trafia do `outputs/tables/player_match_dataset.csv`. Katalog `outputs/` jest miejscem na wygenerowane artefakty, wi?c mo?esz spokojnie nadpisywa? plik po zmianie cech.


In [ ]:
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
player_match_dataset.to_csv(OUTPUT_PATH, index=False, encoding="utf-8")

print("Zapisano:", OUTPUT_PATH)
print("Liczba wierszy:", len(player_match_dataset))


## Co dalej?

Ten dataset jest baz? do wyboru cech. Nast?pny krok to zaw??enie kolumn do najlepszego zestawu, np. 10 cech zawodnika plus kilka zmiennych kontekstowych meczu.
